The following exercises are meant to be solved by gathering the bash commands incrimentally in two scripts, one for ex 1.* the other for ex 2.* 

### Ex 1

1\.a Make a new directory called `students` in your home. Download a csv file with the list of students of this lab from [here](https://www.dropbox.com/s/867rtx3az6e9gm8/LCP_22-23_students.csv) (use the `wget` command) and copy that to `students`. First check whether the file is already there

1\.b Make two new files, one containing the students belonging to PoD, the other to Physics.

1\.c For each letter of the alphabet, count the number of students whose surname starts with that letter. 

1\.d Find out which is the letter with most counts.

1\.e Assume an obvious numbering of the students in the file (first line is 1, second line is 2, etc.), group students "modulo 18", i.e. 1,19,37,.. 2,20,38,.. etc. and put each group in a separate file  

In [ ]:
#!/bin/bash

cd $HOME
rm students/*
rm -d students/
mkdir -p students #makes dir if not existent
if [ ! -f "./students/LCP_22-23_students.csv" ] #if not already present 
then
    wget -v --tries=1 https://www.dropbox.com/s/867rtx3az6e9gm8/LCP_22-23_students.csv --directory-prefix="./students" #import file in ./students
fi
cd students
touch LCP_22-23_PoD_students.csv #create if not existing
touch LCP_22-23_Physics_students.csv
grep "PoD" LCP_22-23_students.csv > LCP_22-23_PoD_students.csv #copy only lines including PoD
grep "Physics" LCP_22-23_students.csv > LCP_22-23_Physics_students.csv
max=0
max_L='A'
for i in {A..Z}
do
    j=`grep -v -e  "^Family" LCP_22-23_students.csv | grep -c "^$i" LCP_22-23_students.csv` # first gives lines without Family Name, second counts starting with A
    echo "$i : $j"
    if [ $j -gt $max ] #compare values, not string
    then
        max=$j
        max_L=$i
    fi
done
echo "Max surname starting letter $max_L with $max entries"
lines=`grep '' -c LCP_22-23_students.csv`
i=2
while [ $i -le $lines ]
do
    let g=($i-1)%18 #real computation for variables needs let or (( ))
    file="Group$g.csv"
    touch $file
    awk NR==$i LCP_22-23_students.csv >> $file #selected line with awk
    let i+=1
done
#spacing is always wrong in bash, unless is necessary

### Ex 2

2.a Make a copy of the file `data.csv` removing the metadata and the commas between numbers; call it `data.txt`

2\.b How many even numbers are there?

2\.c Distinguish the entries on the basis of `sqrt(X^2 + Y^2 + Z^2)` is greater or smaller than `100*sqrt(3)/2`. Count the entries of each of the two groups 

2\.d Make `n` copies of data.txt (with `n` an input parameter of the script), where the i-th copy has all the numbers divided by i (with `1<=i<=n`).

In [ ]:
#!/bin/bash

grep -v '^#' data.csv | sed -e 's/,//g' > data.txt
even=0
for i in `cat data.txt`
do
    if [ $i%2 ] #%2 gives 0 if even, that correspond to true in bash
    then
        let even+=1
    fi
done
echo "even numbers = $even"
m=0
l=0
sigma=$( echo 'scale=6;100*sqrt(3)/2.0' | bc)
FILE="data.txt"
lines=`grep '' -c $FILE`
i=1
while [ $i -le $lines ]
do
    line=`awk NR==$i $FILE`
    IFS=' ' read -r X Y Z x y z <<< "$line"
    d=$( echo "scale=6;sqrt($X*$X+$Y*$Y+$Z*$Z)" | bc)
    #echo $d
    if [ `echo "$d < $sigma" | bc` -eq 0 ]
    then
        let m++
    else
        let l++
    fi
    let i++
done
echo "there are $m of distance grater than $sigma"
echo "there are $l of distance smaller than $sigma"
if [ -z $1 ]
then
    echo "This program requires an input for normalization"
    exit
fi
if [ $1 -lt 1 ]
then
    echo "This program requires an input grater than 1 for normalization"
    exit
fi
for (( i=1; i<=$1; i++ ))
do
    #-v passes i as a variable to awk; cycle over NF Number of Fields; $j content of j field, 
    #check if field equal to number and end ($); then print as float j/i
    awk -v i="$i" '{for(j=1;j<=NF;j++) if($j~/^[0-9]+$/) $j=sprintf("%.1f",$j/i)}1' data.txt > "data$i.csv"
    done
#[0-9]* for zero or more numbers; [0-9][0-9]* one or more; etc. those are patterns with 2 and 3 element resp. \1 \2 \3
# % echo "123 abc" | sed 's/[0-9]*/& &/'
# 123 123 abc
# sed y is like tr